In [7]:
import cv2
import numpy as np
import os
import random
from math import cos, sin, pi

IMG_SIZE = 416
NUM_IMAGES = 50000  
SAVE_DIR = "shapes_dataset"
SHAPES = ["circle", "triangle", "square", "rectangle", "star", "pentagon", "hexagon", "ellipse", "cross"]
CLASS_MAP = {name: idx for idx, name in enumerate(SHAPES)}

os.makedirs(f"{SAVE_DIR}/images", exist_ok=True)
os.makedirs(f"{SAVE_DIR}/labels", exist_ok=True)

def add_noise(img):
    noise = np.random.normal(0, 25, img.shape).astype(np.uint8)
    img = cv2.add(img, noise)
    for _ in range(random.randint(5, 15)):
        pt1 = (random.randint(0, IMG_SIZE), random.randint(0, IMG_SIZE))
        pt2 = (random.randint(0, IMG_SIZE), random.randint(0, IMG_SIZE))
        color = [random.randint(0, 255) for _ in range(3)]
        cv2.line(img, pt1, pt2, color, 1)
    for _ in range(random.randint(50, 100)):
        pt = (random.randint(0, IMG_SIZE), random.randint(0, IMG_SIZE))
        img[pt[1] % IMG_SIZE, pt[0] % IMG_SIZE] = [random.randint(0, 255) for _ in range(3)]
    return img

def rotate_points(pts, center, angle_rad):
    return [[
        int(center[0] + (x - center[0]) * cos(angle_rad) - (y - center[1]) * sin(angle_rad)),
        int(center[1] + (x - center[0]) * sin(angle_rad) + (y - center[1]) * cos(angle_rad))
    ] for x, y in pts]

def draw_shape(img, shape_name):
    center = (random.randint(80, IMG_SIZE - 80), random.randint(80, IMG_SIZE - 80))
    size = random.randint(30, 90)
    color = [random.randint(0, 255) for _ in range(3)]
    angle = random.uniform(0, 2 * pi)
    thickness = -1

    if shape_name == "circle":
        cv2.circle(img, center, size, color, thickness)
        return (center[0] - size, center[1] - size, center[0] + size, center[1] + size)

    def draw_polygon(n_sides):
        return [[
            int(center[0] + size * cos(2 * pi * i / n_sides + angle)),
            int(center[1] + size * sin(2 * pi * i / n_sides + angle))
        ] for i in range(n_sides)]

    if shape_name == "triangle":
        pts = rotate_points([[center[0], center[1] - size],
                             [center[0] - size, center[1] + size],
                             [center[0] + size, center[1] + size]], center, angle)
    elif shape_name == "square":
        pts = rotate_points([[center[0] - size, center[1] - size],
                             [center[0] + size, center[1] - size],
                             [center[0] + size, center[1] + size],
                             [center[0] - size, center[1] + size]], center, angle)
    elif shape_name == "rectangle":
        w, h = size, size // 2
        pts = rotate_points([[center[0] - w, center[1] - h],
                             [center[0] + w, center[1] - h],
                             [center[0] + w, center[1] + h],
                             [center[0] - w, center[1] + h]], center, angle)
    elif shape_name == "star":
        pts = [[
            int(center[0] + (size if i % 2 == 0 else size // 2) * cos(pi / 5 * i + angle)),
            int(center[1] + (size if i % 2 == 0 else size // 2) * sin(pi / 5 * i + angle))
        ] for i in range(10)]
    elif shape_name == "pentagon":
        pts = draw_polygon(5)
    elif shape_name == "hexagon":
        pts = draw_polygon(6)
    elif shape_name == "ellipse":
        axes = (size, size // 2)
        cv2.ellipse(img, center, axes, angle * 180 / pi, 0, 360, color, thickness)
        return (center[0] - axes[0], center[1] - axes[1], center[0] + axes[0], center[1] + axes[1])
    elif shape_name == "cross":
        w = size // 4
        long = size
        pts1 = [[center[0] - w, center[1] - long], [center[0] + w, center[1] - long],
                [center[0] + w, center[1] + long], [center[0] - w, center[1] + long]]
        pts2 = [[center[0] - long, center[1] - w], [center[0] + long, center[1] - w],
                [center[0] + long, center[1] + w], [center[0] - long, center[1] + w]]
        for pts in [pts1, pts2]:
            rotated = np.array(rotate_points(pts, center, angle)).reshape((-1, 1, 2))
            cv2.fillPoly(img, [rotated], color)
        x, y, w, h = cv2.boundingRect(np.array(rotate_points(pts1 + pts2, center, angle)))
        return (x, y, x + w, y + h)

    pts = np.array(pts).reshape((-1, 1, 2))
    cv2.fillPoly(img, [pts], color)
    x, y, w, h = cv2.boundingRect(pts)
    return (x, y, x + w, y + h)

def convert_to_yolo_format(bbox, img_w, img_h):
    x1, y1, x2, y2 = bbox
    x_center = ((x1 + x2) / 2) / img_w
    y_center = ((y1 + y2) / 2) / img_h
    width = (x2 - x1) / img_w
    height = (y2 - y1) / img_h
    return x_center, y_center, width, height

# Generate dataset
for i in range(NUM_IMAGES):
    img = np.ones((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8) * 255
    shape = random.choice(SHAPES)
    bbox = draw_shape(img, shape)
    x, y, w, h = convert_to_yolo_format(bbox, IMG_SIZE, IMG_SIZE)
    class_id = CLASS_MAP[shape]

    img = add_noise(img)

    img_path = f"{SAVE_DIR}/images/image_{i}.jpg"
    label_path = f"{SAVE_DIR}/labels/image_{i}.txt"

    cv2.imwrite(img_path, img)
    with open(label_path, "w") as f:
        f.write(f"{class_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")

    if i % 1000 == 0:
        print(f"[+] {i}/{NUM_IMAGES} generated...")

print(f"[✓] Completed: {NUM_IMAGES} images with {len(SHAPES)} shapes saved to '{SAVE_DIR}' ✅")


[+] 0/50000 generated...
[+] 1000/50000 generated...
[+] 2000/50000 generated...
[+] 3000/50000 generated...
[+] 4000/50000 generated...
[+] 5000/50000 generated...
[+] 6000/50000 generated...
[+] 7000/50000 generated...
[+] 8000/50000 generated...
[+] 9000/50000 generated...
[+] 10000/50000 generated...
[+] 11000/50000 generated...
[+] 12000/50000 generated...
[+] 13000/50000 generated...
[+] 14000/50000 generated...
[+] 15000/50000 generated...
[+] 16000/50000 generated...
[+] 17000/50000 generated...
[+] 18000/50000 generated...
[+] 19000/50000 generated...
[+] 20000/50000 generated...
[+] 21000/50000 generated...
[+] 22000/50000 generated...
[+] 23000/50000 generated...
[+] 24000/50000 generated...
[+] 25000/50000 generated...
[+] 26000/50000 generated...
[+] 27000/50000 generated...
[+] 28000/50000 generated...
[+] 29000/50000 generated...
[+] 30000/50000 generated...
[+] 31000/50000 generated...
[+] 32000/50000 generated...
[+] 33000/50000 generated...
[+] 34000/50000 generated..

In [8]:
import os
import cv2
import random
import matplotlib.pyplot as plt
from ultralytics import YOLO

In [11]:
import os
import random
import shutil

# Paths
dataset_dir = "D:/PROJECTS/SmartMirror/shapes_dataset"
image_dir = os.path.join(dataset_dir, "images")
label_dir = os.path.join(dataset_dir, "labels")

train_image_dir = os.path.join(dataset_dir, "images/train")
val_image_dir = os.path.join(dataset_dir, "images/val")
train_label_dir = os.path.join(dataset_dir, "labels/train")
val_label_dir = os.path.join(dataset_dir, "labels/val")

# Make directories
os.makedirs(train_image_dir, exist_ok=True)
os.makedirs(val_image_dir, exist_ok=True)
os.makedirs(train_label_dir, exist_ok=True)
os.makedirs(val_label_dir, exist_ok=True)

# Get all image filenames
all_images = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png'))]
random.shuffle(all_images)

# Split
split_index = int(0.8 * len(all_images))
train_images = all_images[:split_index]
val_images = all_images[split_index:]

# Move files
for img in train_images:
    lbl = img.rsplit('.', 1)[0] + '.txt'
    shutil.copy(os.path.join(image_dir, img), os.path.join(train_image_dir, img))
    shutil.copy(os.path.join(label_dir, lbl), os.path.join(train_label_dir, lbl))

for img in val_images:
    lbl = img.rsplit('.', 1)[0] + '.txt'
    shutil.copy(os.path.join(image_dir, img), os.path.join(val_image_dir, img))
    shutil.copy(os.path.join(label_dir, lbl), os.path.join(val_label_dir, lbl))

print("✅ Done splitting dataset.")

✅ Done splitting dataset.


In [12]:
dataset_dir = "shapes_dataset"
assert os.path.exists(f"{dataset_dir}/images/train"), "Missing training images"
assert os.path.exists(f"{dataset_dir}/labels/train"), "Missing training labels"

In [ ]:
model = YOLO("yolov8n.pt")
model.train(data="shapes.yaml", epochs=30, imgsz=416, batch=8)

New https://pypi.org/project/ultralytics/8.3.113 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.104  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)
engine\trainer: task=detect, mode=train, model=yolov8n.pt, data=shapes.yaml, epochs=30, time=None, patience=100, batch=8, imgsz=416, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_fr

train: Scanning D:\PROJECTS\SmartMirror\shapes_dataset\labels\train... 40000 images, 0 backgrounds, 0 corrupt: 100%|██████████|


train: New cache created: D:\PROJECTS\SmartMirror\shapes_dataset\labels\train.cache


val: Scanning D:\PROJECTS\SmartMirror\shapes_dataset\labels\val... 10000 images, 0 backgrounds, 0 corrupt: 100%|██████████| 100


val: New cache created: D:\PROJECTS\SmartMirror\shapes_dataset\labels\val.cache
Plotting labels to runs\detect\train\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: SGD(lr=0.01, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)


In [ ]:
# Load the best model
from ultralytics import YOLO

# Load the trained best weights
model = YOLO('runs/detect/train/weights/best.pt')

# Export to ONNX (good for Pi 5)
model.export(format='onnx', dynamic=True)

# Export to TorchScript (alternative for PyTorch Mobile)
model.export(format='torchscript')

print("Model exported in ONNX and TorchScript formats.")

In [ ]:
def show_random_annotated_image(img_dir, label_dir):
    images = os.listdir(img_dir)
    img_file = random.choice(images)
    img_path = os.path.join(img_dir, img_file)
    label_path = os.path.join(label_dir, img_file.replace(".jpg", ".txt"))

    image = cv2.imread(img_path)
    h, w = image.shape[:2]

    with open(label_path, "r") as f:
        for line in f.readlines():
            cls, x, y, box_w, box_h = map(float, line.strip().split())
            x1 = int((x - box_w/2) * w)
            y1 = int((y - box_h/2) * h)
            x2 = int((x + box_w/2) * w)
            y2 = int((y + box_h/2) * h)
            cv2.rectangle(image, (x1, y1), (x2, y2), (0,255,0), 2)
            cv2.putText(image, str(int(cls)), (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)

    plt.figure(figsize=(6,6))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title(f"Sample Annotated Image - {img_file}")
    plt.show()

# Show sample from training set
show_random_annotated_image(f"{dataset_dir}/images/train", f"{dataset_dir}/labels/train")


<Figure size 600x600 with 1 Axes>

In [ ]:
# Show results in notebook
model.val()  # Evaluate to update metrics

results_dir = model.train_args.save_dir
metrics_plot = os.path.join(results_dir, "results.png")

# Display the metrics plot
if os.path.exists(metrics_plot):
    img = cv2.imread(metrics_plot)
    plt.figure(figsize=(12,6))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title("📈 Training Metrics (Loss, Precision, Recall, mAP)")
    plt.show()
else:
    print("No metrics plot found.")


Ultralytics 8.3.104  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)
Model summary (fused): 72 layers, 3,007,598 parameters, 0 gradients, 8.1 GFLOPs


val: Scanning D:\PROJECTS\SmartMirror\shapes_dataset\labels\val.cache... 4737 images, 0 backgrounds, 0 corrupt: 100%|███
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 593/593 [00:1


                   all       4737       4737          1      0.999      0.995      0.995
                circle        942        942          1      0.999      0.995      0.995
              triangle        968        968          1      0.999      0.995      0.995
                square        906        906          1      0.999      0.995      0.995
             rectangle        977        977          1          1      0.995      0.995
                  star        944        944          1          1      0.995      0.995
Speed: 0.1ms preprocess, 1.0ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to runs\detect\train62


AttributeError: 'DetectionModel' object has no attribute 'train_args'

In [ ]:
# Pick a random validation image
val_img_path = random.choice(os.listdir(f"{dataset_dir}/images/val"))
img_path = os.path.join(dataset_dir, "images/val", val_img_path)

# Run inference
results = model.predict(img_path, conf=0.25)

# Show the image with predicted boxes
results[0].show()  # This works in most notebook environments



image 1/1 D:\PROJECTS\SmartMirror\shapes_dataset\images\val\image_8910.jpg: 416x416 1 circle, 7.5ms
Speed: 0.9ms preprocess, 7.5ms inference, 2.3ms postprocess per image at shape (1, 3, 416, 416)


In [ ]:
# Run inference
results = model.predict(img_path, conf=0.25)

# Show the image with predicted boxes
results[0].show()  


image 1/1 D:\PROJECTS\SmartMirror\TEST\T3.jpg: 384x640 2 persons, 1 book, 6.0ms
Speed: 1.1ms preprocess, 6.0ms inference, 3.5ms postprocess per image at shape (1, 3, 384, 640)
